# Parlay pricing: combined odds and the correlation trap

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JacobiusMakes/parlayapi-notebooks/blob/main/05-parlay-pricing.ipynb)

A parlay pays only if every leg wins, so the legs' decimal odds multiply. The
multiplication is exact and takes four lines of Python; the interesting parts are
what compounds along with it (the vig) and what silently breaks it (correlation).
This notebook follows the same conventions as ParlayAPI's
[parlay calculator](https://parlay-api.com/tools/parlay-calculator), builds a
parlay from live odds, and cross-checks the result against the API's keyless
parlay pricer.

Works with no key. A [free key](https://parlay-api.com/signup) adds books, markets, and the
correlation-aware `/v1/sgp/price` endpoint.

In [1]:
# ---- Config: paste your API key between the quotes ----
# Get a free key at https://parlay-api.com/signup (free tier, no card required).
# Leave it empty to run against the keyless demo endpoint instead.
API_KEY = ""

import os
API_KEY = (API_KEY or os.environ.get("PARLAYAPI_KEY", "")).strip()
BASE_URL = "https://parlay-api.com"
SPORT = "baseball_mlb"  # also try: basketball_nba, americanfootball_nfl, icehockey_nhl, soccer_epl

if API_KEY:
    print("API key set: using the full keyed endpoints.")
else:
    print("No API key set: falling back to the keyless demo endpoint.")
    print("The demo serves the first 5 events per sport, moneyline (h2h) only,")
    print("capped at 60 requests per hour. Paste a free key above for all events,")
    print("all 30+ books, and every market:", "https://parlay-api.com/signup")

No API key set: falling back to the keyless demo endpoint.
The demo serves the first 5 events per sport, moneyline (h2h) only,
capped at 60 requests per hour. Paste a free key above for all events,
all 30+ books, and every market: https://parlay-api.com/signup


In [2]:
# American <-> decimal conversions, matching the conventions used by
# https://parlay-api.com/tools/no-vig-calculator and /tools/parlay-calculator.

def american_to_decimal(a):
    """+150 -> 2.5, -110 -> 1.9091. Valid American odds are >= +100 or <= -100."""
    a = float(a)
    if abs(a) < 100:
        raise ValueError(f"{a} is not a valid American price (must be >= +100 or <= -100)")
    if a > 0:
        return 1 + a / 100
    return 1 + 100 / (-a)

def decimal_to_american(d):
    """2.5 -> +150, 1.9091 -> -110 (rounded to the nearest integer)."""
    d = float(d)
    if d <= 1:
        raise ValueError(f"decimal odds must be > 1, got {d}")
    if d >= 2:
        return round((d - 1) * 100)
    return round(-100 / (d - 1))

def implied_prob(decimal_odds):
    """Implied win probability of decimal odds (includes the vig)."""
    return 1.0 / float(decimal_odds)

def fmt_american(a):
    return ("+" if a > 0 else "") + str(int(a))

## The core math

Convert every leg to decimal odds, multiply the decimals, multiply by the stake:

```
combined_dec = dec_1 * dec_2 * ... * dec_n
payout       = stake * combined_dec      (includes your stake back)
profit       = payout - stake
implied_prob = 1 / combined_dec
```

In [3]:
from functools import reduce

def parlay(american_legs, stake=100.0):
    decs = [american_to_decimal(a) for a in american_legs]
    combined = reduce(lambda x, y: x * y, decs, 1.0)
    return {
        "legs": american_legs,
        "combined_decimal": combined,
        "combined_american": decimal_to_american(combined),
        "implied_prob": 1.0 / combined,
        "stake": stake,
        "payout": stake * combined,
        "profit": stake * combined - stake,
    }

p = parlay([-110, -110, -110], stake=100)
print(f"three legs at -110, $100 stake:")
print(f"  combined decimal  {p['combined_decimal']:.4f}")
print(f"  combined American {fmt_american(p['combined_american'])}")
print(f"  implied prob      {p['implied_prob']*100:.2f}%")
print(f"  payout ${p['payout']:.2f}  (profit ${p['profit']:.2f})")

# The canonical case from the site parlay calculator:
assert abs(p["combined_decimal"] - 6.9579) < 1e-3
assert p["combined_american"] == 596
assert abs(p["payout"] - 695.79) < 0.01
assert abs(p["implied_prob"] - 0.1437) < 1e-3
print("matches the /tools/parlay-calculator canonical case (+596, $695.79, 14.37%)")

three legs at -110, $100 stake:
  combined decimal  6.9579
  combined American +596
  implied prob      14.37%
  payout $695.79  (profit $595.79)
matches the /tools/parlay-calculator canonical case (+596, $695.79, 14.37%)


## The vig compounds with every leg

Each -110 price hides roughly a 2.4% edge for the book (its fair value is +100 in
a balanced market, per notebook 02). Because parlay legs multiply, that edge
multiplies too: if each -110 leg's true chance is 50%, each leg returns
`0.50 * 1.9091 = 0.9545` of the stake in expectation, and n legs return
`0.9545 ** n`.

In [4]:
leg_dec = american_to_decimal(-110)
true_p = 0.50  # assume the balanced-market fair value
per_leg_return = true_p * leg_dec

print("legs | expected return per $1 | expected loss")
for n in range(1, 7):
    er = per_leg_return ** n
    print(f"  {n}  |        {er:.4f}          |   {(1-er)*100:5.1f}%")

three = per_leg_return ** 3
assert abs(three - 0.870) < 5e-4
print("\nthree legs of -110 at a true 50% each: expected return 0.870,")
print("about a 13% expected loss, exactly as the site calculator's FAQ works out.")

legs | expected return per $1 | expected loss
  1  |        0.9545          |     4.5%
  2  |        0.9112          |     8.9%
  3  |        0.8697          |    13.0%
  4  |        0.8302          |    17.0%
  5  |        0.7925          |    20.8%
  6  |        0.7564          |    24.4%

three legs of -110 at a true 50% each: expected return 0.870,
about a 13% expected loss, exactly as the site calculator's FAQ works out.


## Build a parlay from the live board

Two legs, one from each of the first two events on the slate, taking each side's
best available price across books (line shopping matters even more in parlays,
because the improvement multiplies).

In [5]:
import requests

def fetch_odds(sport=None, markets="h2h,spreads,totals", odds_format="american"):
    """Fetch current odds as a list of event dicts.

    Keyed:   GET /v1/sports/{sport}/odds returns a bare JSON array of events
             (the-odds-api compatible shape).
    Keyless: GET /v1/try/{sport}/odds returns a demo envelope instead: the
             events are nested under the "events" key, next to demo metadata
             like demo_message and demo_remaining_hour. The two shapes are
             NOT the same at the top level, so we unwrap here.

    Each event: id, home_team, away_team, commence_time, and
    bookmakers[] -> markets[] -> outcomes[] with American prices by default.
    """
    sport = sport or SPORT
    if API_KEY:
        resp = requests.get(
            f"{BASE_URL}/v1/sports/{sport}/odds",
            params={"markets": markets, "oddsFormat": odds_format},
            headers={"X-API-Key": API_KEY},
            timeout=30,
        )
        resp.raise_for_status()
        return resp.json()
    resp = requests.get(f"{BASE_URL}/v1/try/{sport}/odds", timeout=30)
    resp.raise_for_status()
    payload = resp.json()
    # Demo envelope: {"demo": true, "demo_message": "...", "events": [...]}
    return payload.get("events", [])

In [6]:
try:
    events = fetch_odds(markets="h2h")
except Exception as exc:
    events = []
    print(f"Fetch failed ({exc}). Check your connection or key and re-run this cell.")
legs = []
for ev in events[:2]:
    quotes = []
    for bm in ev.get("bookmakers", []):
        for mkt in bm.get("markets", []):
            if mkt["key"] != "h2h":
                continue
            for out in mkt["outcomes"]:
                if out["name"] == ev["home_team"]:
                    quotes.append((bm["key"], out["price"],
                                   american_to_decimal(out["price"])))
    if not quotes:
        continue
    # Outlier guard: a thin exchange-style listing can post a price no real
    # book will honor, and a naive max would pick exactly that row. Drop
    # anything more than 15% above the median decimal before shopping.
    med = sorted(q[2] for q in quotes)[len(quotes) // 2]
    sane = [q for q in quotes if q[2] <= med * 1.15] or quotes
    book, price, _ = max(sane, key=lambda q: q[2])
    legs.append({"event": f"{ev['away_team']} at {ev['home_team']}",
                 "side": ev["home_team"], "book": book, "price": price})

if len(legs) < 2:
    print("Fewer than two priced events on the board right now; re-run later")
    print("or change SPORT in the config cell.")
else:
    for leg in legs:
        print(f"leg: {leg['side']} {fmt_american(leg['price'])} "
              f"(best price, at {leg['book']}) in {leg['event']}")
    p = parlay([leg["price"] for leg in legs], stake=50)
    print(f"\ncombined: {p['combined_decimal']:.4f} decimal "
          f"= {fmt_american(p['combined_american'])} American")
    print(f"$50 pays ${p['payout']:.2f} if both legs win "
          f"(implied {p['implied_prob']*100:.1f}%)")

leg: Atlanta Braves -208 (best price, at novig) in Colorado Rockies at Atlanta Braves
leg: Chicago Cubs -140 (best price, at fliff) in Cincinnati Reds at Chicago Cubs

combined: 2.5385 decimal = +154 American
$50 pays $126.92 if both legs win (implied 39.4%)


## Cross-check against the API's parlay pricer

`POST /v1/try/sgp/price` is keyless (up to 4 legs, 60 requests per hour) and
returns the same independent-baseline price computed above, as the product of the
legs' implied probabilities converted back to American. If our math and the API
disagree, one of us is wrong.

In [7]:
import requests

check_legs = [leg["price"] for leg in legs] if len(legs) >= 2 else [-110, -110, -110]
try:
    r = requests.post(f"{BASE_URL}/v1/try/sgp/price",
                      json={"legs": [{"price": p_} for p_ in check_legs]},
                      timeout=30)
except requests.RequestException as exc:
    r = None
    print(f"Pricer endpoint unreachable right now ({exc}); skipping the cross-check.")
if r is not None and r.ok:
    api = r.json()["independent_baseline"]
    ours = parlay(check_legs)
    print(f"legs {check_legs}")
    print(f"  our combined:  {ours['combined_decimal']:.3f} decimal, "
          f"{fmt_american(ours['combined_american'])}")
    print(f"  API combined:  {api['decimal_price']} decimal, "
          f"{fmt_american(api['american_price'])}")
    assert abs(ours["combined_decimal"] - api["decimal_price"]) < 0.01
    print("  agreement within rounding: good")
elif r is not None:
    print(f"Pricer endpoint returned HTTP {r.status_code}; skipping the cross-check.")

legs [-208, -140]
  our combined:  2.538 decimal, +154
  API combined:  2.538 decimal, +154
  agreement within rounding: good


## The correlation trap

Everything above multiplies probabilities, which is only valid when the legs are
**independent**. Same-game legs usually are not:

- A quarterback's passing yards over and his team's win are positively
  correlated: parlaying them at the independent price underpays you if the book
  priced them independently, so books do not; they price the correlation in.
- A favorite's moneyline and the game total can be correlated through game
  script (a blowout tends to mean more possessions for one side).
- Even cross-game legs can correlate (weather systems, shared referees are weak
  examples; same-division doubleheaders less so).

Practical rules:

1. For cross-game parlays of unrelated events, independent multiplication is the
   right baseline, and everything in this notebook applies as-is.
2. For same-game parlays, the independent product is only a **baseline**. Real
   SGP prices move away from it, in either direction, depending on the
   correlation between the legs. That is why the keyless endpoint labels itself
   an independent baseline and why the keyed `/v1/sgp/price` endpoint exists: it
   applies literature-based correlation adjustments per leg pair. See
   [the docs](https://parlay-api.com/docs).
3. The vig compounding from earlier applies either way: every leg you add
   multiplies the book's margin into your price. Devig each leg (notebook 02)
   before deciding a parlay is worth it.

---

**More ParlayAPI resources**

- Docs: [parlay-api.com/docs](https://parlay-api.com/docs)
- Free API key (no card): [parlay-api.com/signup](https://parlay-api.com/signup)
- Browser calculators the math here matches: [no-vig](https://parlay-api.com/tools/no-vig-calculator), [parlay](https://parlay-api.com/tools/parlay-calculator), [EV](https://parlay-api.com/tools/ev-calculator)
- The rest of this series: [github.com/JacobiusMakes/parlayapi-notebooks](https://github.com/JacobiusMakes/parlayapi-notebooks)

These notebooks are for research and education. Nothing here is betting advice.